# Capstone Project 2: Predicting Movie Profitability

# Part 4: Pre-processing & Training Data Development

## 1. Introduction

In this section, we prepare the data for modeling. This involves handling categorical variables (One-Hot Encoding), splitting the data into training and testing sets, and scaling numerical features to ensure our machine learning algorithms perform optimally.

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
# Load the cleaned data
df = pd.read_csv('movie_data_cleaned.csv')

# Dataset overview
print(f"Data Shape: {df.shape}")
display(df.head())

Data Shape: (6420, 20)


,Movie Name,Release Date,Production Budget (USD),Domestic Gross (USD),Worldwide Gross (USD),MPAA Rating,Running Time (minutes),Franchise,Keywords,Source,Genre,Production Method,Creative Type,Production/Financing Companies,Production Countries,Languages,ROI,is_profitable,Release Year,Release Month
0,Star Wars Ep. VII: The Force Awakens,2015-12-16,533200000,936662225,2056046835,PG-13,136.0,1,"Space Opera,Good vs. Evil,Delayed Sequel,Inter...",Original Screenplay,Adventure,"Animation,Live Action",Science Fiction,"Lucasfilm,Bad Robot",United States,English,3.856052,1,2015,12
1,Avatar: The Way of Water,2022-12-09,460000000,684075767,2315589775,PG-13,190.0,1,"Action Adventure,Delayed Sequel,Humans as Alie...",Original Screenplay,Action,"Animation,Live Action",Science Fiction,"Lightstorm Entertainment,20th Century Studios,...",United States,English,5.033891,1,2022,12
2,Indiana Jones and the Dial of Destiny,2023-06-28,402300000,174480468,383963057,PG-13,142.0,1,"1960s,Space Program,Nazis Outside of World War...",Original Screenplay,Adventure,Live Action,Historical Fiction,"Lucasfilm,Walt Disney Pictures,Paramount Pictures",United States,English,0.954420,0,2023,6
3,Avengers: Endgame,2019-04-23,400000000,858373000,2748242781,PG-13,181.0,1,"Ensemble,Marvel Comics,Animal Lead,Non-Chronol...",Based on Comic/Graphic Novel,Action,"Animation,Live Action",Super Hero,Marvel Studios,United States,English,6.870607,1,2019,4
4,Mission: Impossible—The Final Reckoning,2025-05-21,400000000,0,0,Unrated,105.0,0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.000000,0,2025,5


## 2. Feature Selection

We are creating a alternative dataset for trainning model and dropping columns that are too complex for model trainning.

In [6]:
# Drop columns that won't be used for prediction
cols_to_drop = ['Movie Name', 'Release Date', 'ROI', 'Keywords', 'Production/Financing Companies', 'Production Countries', 'Languages']

df_model = df.drop(columns=cols_to_drop)

# Verify the remaining columns
print(f"Original shape: {df.shape}")
print(f"Modeling shape: {df_model.shape}")
display(df_model.head())


Original shape: (6420, 20)
Modeling shape: (6420, 13)


,Production Budget (USD),Domestic Gross (USD),Worldwide Gross (USD),MPAA Rating,Running Time (minutes),Franchise,Source,Genre,Production Method,Creative Type,is_profitable,Release Year,Release Month
0,533200000,936662225,2056046835,PG-13,136.0,1,Original Screenplay,Adventure,"Animation,Live Action",Science Fiction,1,2015,12
1,460000000,684075767,2315589775,PG-13,190.0,1,Original Screenplay,Action,"Animation,Live Action",Science Fiction,1,2022,12
2,402300000,174480468,383963057,PG-13,142.0,1,Original Screenplay,Adventure,Live Action,Historical Fiction,0,2023,6
3,400000000,858373000,2748242781,PG-13,181.0,1,Based on Comic/Graphic Novel,Action,"Animation,Live Action",Super Hero,1,2019,4
4,400000000,0,0,Unrated,105.0,0,Unknown,Unknown,Unknown,Unknown,0,2025,5


## 3. Creating Dummy Variables (One-Hot Encoding)

We convert categorical variables (`Genre`, `MPAA Rating`, `Source`, etc.) into numeric binary columns.

In [7]:
# Identify categorical columns to encode
categorical_cols = ['MPAA Rating', 'Source', 'Genre', 'Production Method', 'Creative Type']

# Create dummy variables
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print(f"Shape after Encoding: {df_encoded.shape}")
display(df_encoded.head())

Shape after Encoding: (6420, 92)


,Production Budget (USD),Domestic Gross (USD),Worldwide Gross (USD),Running Time (minutes),Franchise,is_profitable,Release Year,Release Month,MPAA Rating_M/PG,MPAA Rating_NC-17,...,Creative Type_Contemporary Fiction,Creative Type_Dramatization,Creative Type_Factual,Creative Type_Fantasy,Creative Type_Historical Fiction,Creative Type_Kids Fiction,Creative Type_Multiple Creative Types,Creative Type_Science Fiction,Creative Type_Super Hero,Creative Type_Unknown
0,533200000,936662225,2056046835,136.0,1,1,2015,12,False,False,...,False,False,False,False,False,False,False,True,False,False
1,460000000,684075767,2315589775,190.0,1,1,2022,12,False,False,...,False,False,False,False,False,False,False,True,False,False
2,402300000,174480468,383963057,142.0,1,0,2023,6,False,False,...,False,False,False,False,True,False,False,False,False,False
3,400000000,858373000,2748242781,181.0,1,1,2019,4,False,False,...,False,False,False,False,False,False,False,False,True,False
4,400000000,0,0,105.0,0,0,2025,5,False,False,...,False,False,False,False,False,False,False,False,False,True


## 4. Train/Test Split

We split the data into **features (X)** and the **target (y)**. Then, we divide the data into a training set (80%) and a testing set (20%) to evaluate our model's performance on unseen data.

In [8]:
# Define X (Features) and y (Target)
X = df_encoded.drop(columns=['is_profitable'])
y = df_encoded['is_profitable']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training Set shape: {X_train.shape}")
print(f"Testing Set shape: {X_test.shape}")

Training Set shape: (5136, 91)
Testing Set shape: (1284, 91)


## 5. Feature Scaling

We use `StandardScaler` to normalize numerical features.

In [9]:
# Initialize the Scaler
scaler = StandardScaler()

# Identify columns to scale
num_cols = ['Production Budget (USD)', 'Running Time (minutes)', 'Release Year', 'Release Month']

# Fit on Train, Transform on Train and Test
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Verify scaling
print("Training Data Mean (After Scaling):")
print(X_train[num_cols].mean())

Training Data Mean (After Scaling):
Production Budget (USD)   -3.389466e-17
Running Time (minutes)     1.203606e-16
Release Year               3.857766e-15
Release Month              1.369621e-16
dtype: float64
